# 📊 Day 1 Homework Grading System: Data Engineering Pipeline
## For instructors

---

### How to use
1. Run the Setup cell once.
2. Paste one student's GitHub link → run grading.
3. Results will be **appended automatically** to `day1_scores.csv` and `day1_scores.json`.
4. Change the link → run again for the next student (no restart needed).

### Scoring rubric (10 points)

| Step | Max points | Criteria |
|---------|-----------|-------|
| 1. MD5 Duplicate | 2 | Correct duplicate files + correct hash |
| 2. Chunking | 2 | Correct chunk count + correct content in chunk 1 |
| 3. Embedding + Similarity | 3 | Correct score + reasonable explanation |
| 4. Qdrant Retrieval | 3 | Correct score + high-quality summary |



## 📦 Setup (Run )

In [ ]:
%%time
!pip install -q google-genai requests nbformat langchain-text-splitters

import hashlib, os, json, random, re, csv, requests
from datetime import datetime
import nbformat
from google import genai

# Gemini API
try:
 from google.colab import userdata
 GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
 print('✅ API Key Colab Secrets')
except Exception:
 GOOGLE_API_KEY = input('🔑 Gemini API Key: ')

client = genai.Client(api_key=GOOGLE_API_KEY)
MODEL = 'gemini-2.5-pro'

# (append mode)
SCORES_CSV = 'day1_scores.csv'
SCORES_JSON = 'day1_scores.json'

# CSV header 
if not os.path.exists(SCORES_CSV):
 with open(SCORES_CSV, 'w', encoding='utf-8', newline='') as f:
 writer = csv.writer(f)
 writer.writerow(['Timestamp', 'Student Name', 'Student ID', 'Phone', 'LINE ID',
 'Step1', 'Step2', 'Step3', 'Step4', 'Total',
 'AI Suspected', 'Feedback', 'GitHub URL'])
 print(f'📄 {SCORES_CSV} ')
else:
 # 
 with open(SCORES_CSV, 'r') as f:
 existing = sum(1 for _ in f) - 1
 print(f'📄 {SCORES_CSV} {existing} ')

# JSON 
if not os.path.exists(SCORES_JSON):
 with open(SCORES_JSON, 'w') as f:
 json.dump([], f)

print(f'✅ Setup | Model: {MODEL}')

## 🔧 (Run )

In [ ]:
def generate_expected_answers(student_id):
 """"""
 from langchain_text_splitters import RecursiveCharacterTextSplitter
 
 random.seed(int(hashlib.md5(student_id.encode()).hexdigest()[:8], 16))
 
 all_paragraphs = [
 ' Machine Learning ',
 'Deep Learning Machine Learning Neural Network ',
 'Natural Language Processing NLP ',
 'Retrieval Augmented Generation RAG LLM ',
 'Vector Database Embedding Vector ',
 'Text Embedding Vector ',
 'Transformer Neural Network Attention GPT BERT Gemini',
 'Prompt Engineering Prompt LLM Prompt ',
 'Fine-tuning ',
 'Tokenization Token ',
 'Chunking Embedding Fixed-size Recursive Semantic Chunking',
 'Cosine Similarity Vector Vector 1 0 NLP Information Retrieval',
 ]
 
 random.shuffle(all_paragraphs)
 selected = all_paragraphs[:8]
 duplicate_idx = random.randint(0, 5)
 selected.append(selected[duplicate_idx])
 
 hashes = {}
 dup_files = None
 dup_hash = None
 for i, para in enumerate(selected):
 h = hashlib.md5(para.encode()).hexdigest()
 fname = f'doc_{i+1}.txt'
 if h in hashes:
 dup_files = (hashes[h], fname)
 dup_hash = h
 else:
 hashes[h] = fname
 
 unique_texts = []
 seen = set()
 for para in selected:
 h = hashlib.md5(para.encode()).hexdigest()
 if h not in seen:
 unique_texts.append(para)
 seen.add(h)
 
 all_text = '\n\n'.join(unique_texts)
 splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30)
 chunks = splitter.split_text(all_text)
 
 return {
 'student_id': student_id,
 'dup_files': dup_files,
 'dup_hash': dup_hash,
 'files_after_dedup': len(unique_texts),
 'num_chunks': len(chunks),
 'chunk_1_content': chunks[0] if chunks else '',
 'min_chunk_len': min(len(c) for c in chunks) if chunks else 0,
 'verification_code': hashlib.sha256(f'{student_id}_day1_hw'.encode()).hexdigest()[:12],
 }


def fetch_notebook(github_url):
 """ notebook + GitHub"""
 raw_url = github_url.replace('github.com', 'raw.githubusercontent.com').replace('/blob/', '/')
 print(f'📥 : {raw_url}')
 resp = requests.get(raw_url)
 resp.raise_for_status()
 nb = nbformat.reads(resp.text, as_version=4)
 
 info = {'student_name': '', 'student_id': '', 'phone': '', 'line_id': ''}
 full_content = ''
 
 for i, cell in enumerate(nb.cells):
 if cell.cell_type == 'code':
 for key, var in [('student_id','STUDENT_ID'), ('student_name','STUDENT_NAME'),
 ('phone','PHONE'), ('line_id','LINE_ID')]:
 m = re.search(rf"{var}\s*=\s*['\"]([^'\"]+)['\"]", cell.source)
 if m:
 info[key] = m.group(1)
 
 ct = cell.cell_type.upper()
 full_content += f'\n\n=== CELL {i} ({ct}) ===\n{cell.source}'
 if hasattr(cell, 'outputs'):
 for out in cell.outputs:
 if hasattr(out, 'text'):
 full_content += f'\n--- OUTPUT ---\n{out.text}'
 elif hasattr(out, 'data'):
 for mime, data in out.data.items():
 if 'text' in mime:
 txt = data if isinstance(data, str) else '\n'.join(data)
 full_content += f'\n--- OUTPUT ---\n{txt}'
 
 return info, full_content


def grade_with_gemini(student_id, notebook_content, expected):
 """ Gemini 2.5 Pro """
 prompt = f''' AI/RAG 

# 
- : {expected["dup_files"]}, MD5: {expected["dup_hash"]}, {expected["files_after_dedup"]} 
- chunks: {expected["num_chunks"]}, chunk : {expected["min_chunk_len"]} 
- chunk (80 ): "{expected["chunk_1_content"][:80]}..."
- Verification: {expected["verification_code"]}

# Notebook
{notebook_content[:15000]}

# 
## Step 1: MD5 (2 ) — 1 , 0.5 hash , 0.5 
## Step 2: Chunking (2 ) — 1 chunks , 0.5 chunk 1 , 0.5 chunk 
## Step 3: Embedding (3 ) — 1 multilingual-e5-large, 1 cosine similarity+score, 1 
## Step 4: Qdrant (3 ) — 1 collection+upsert, 1 Top-3 scores, 1 

 JSON:
{{"step1_score":0.0,"step1_feedback":"...","step2_score":0.0,"step2_feedback":"...","step3_score":0.0,"step3_feedback":"...","step4_score":0.0,"step4_feedback":"...","total_score":0.0,"overall_feedback":"...","is_ai_suspected":false,"ai_reason":"..."}}

 feedback 
 code/output 0 
 AI is_ai_suspected=true
'''
 resp = client.models.generate_content(
 model=MODEL, contents=prompt,
 config=genai.types.GenerateContentConfig(temperature=0.1, response_mime_type='application/json')
 )
 return json.loads(resp.text)


def append_result(info, grade, github_url):
 """Append CSV + JSON"""
 ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
 
 # Append CSV
 with open(SCORES_CSV, 'a', encoding='utf-8', newline='') as f:
 writer = csv.writer(f)
 writer.writerow([
 ts, info['student_name'], info['student_id'], info['phone'], info['line_id'],
 grade['step1_score'], grade['step2_score'], grade['step3_score'], grade['step4_score'],
 grade['total_score'], grade.get('is_ai_suspected', False),
 grade['overall_feedback'], github_url
 ])
 
 # Append JSON
 with open(SCORES_JSON, 'r') as f:
 all_results = json.load(f)
 grade.update({'timestamp': ts, 'github_url': github_url, **info})
 all_results.append(grade)
 with open(SCORES_JSON, 'w', encoding='utf-8') as f:
 json.dump(all_results, f, ensure_ascii=False, indent=2)
 
 # 
 print(f'💾 ( {len(all_results)} )')


print('✅ ')

---
## ▶️ ()

 `GITHUB_URL` **Run cell ** 
 **append** `day1_scores.csv` 

In [ ]:
assert GITHUB_URL, '❌ GitHub URL!'

# 1. notebook
info, content = fetch_notebook(GITHUB_URL)
print(f'👤 {info["student_name"]} ({info["student_id"]})')
print(f'📱 {info["phone"]} | 💬 {info["line_id"]}')

# 🔍 
already_graded = False
if os.path.exists(SCORES_JSON):
 with open(SCORES_JSON, 'r') as f:
 prev = json.load(f)
 for p in prev:
 if p.get('student_id') == info['student_id']:
 already_graded = True
 print(f'⚠️ {info["student_id"]} (: {p.get("total_score", "?")})')
 break

if already_graded:
 overwrite = input('🔄 overwrite ? (y/n): ').strip().lower()
 if overwrite != 'y':
 print('⏭️ ')
 raise SystemExit()
 # 
 prev = [p for p in prev if p.get('student_id') != info['student_id']]
 with open(SCORES_JSON, 'w', encoding='utf-8') as f:
 json.dump(prev, f, ensure_ascii=False, indent=2)
 # CSV 
 import csv
 rows = []
 with open(SCORES_CSV, 'r', encoding='utf-8') as f:
 reader = csv.reader(f)
 rows = list(reader)
 with open(SCORES_CSV, 'w', encoding='utf-8', newline='') as f:
 writer = csv.writer(f)
 for row in rows:
 if len(row) > 2 and row[2] == info['student_id']:
 continue
 writer.writerow(row)
 print(f'🗑️ {info["student_id"]} ')

# 2. 
expected = generate_expected_answers(info['student_id'])
print(f'🔑 : dup={expected["dup_files"]}, chunks={expected["num_chunks"]}')

# 3. Gemini
print(f'🤖 {MODEL}...')
grade = grade_with_gemini(info['student_id'], content, expected)

# 4. 
print(f'\n{"="*60}')
print(f'📊 : {info["student_name"]} ({info["student_id"]})')
print(f'{"="*60}')
print(f' 1 (MD5): {grade["step1_score"]}/2 — {grade["step1_feedback"]}')
print(f' 2 (Chunking): {grade["step2_score"]}/2 — {grade["step2_feedback"]}')
print(f' 3 (Embedding): {grade["step3_score"]}/3 — {grade["step3_feedback"]}')
print(f' 4 (Qdrant): {grade["step4_score"]}/3 — {grade["step4_feedback"]}')
print(f' {"─"*56}')
print(f' 🏆 : {grade["total_score"]}/10')
print(f' 💬 {grade["overall_feedback"]}')
if grade.get('is_ai_suspected'):
 print(f' ⚠️ AI: {grade["ai_reason"]}')

# 5. Append file
append_result(info, grade, GITHUB_URL)

## 📊 View All Scores


In [ ]:
# 
import csv

with open(SCORES_CSV, 'r', encoding='utf-8') as f:
 reader = csv.reader(f)
 header = next(reader)
 rows = list(reader)

print(f'📊 {len(rows)} \n')
print(f'{"#":>3} {"":<20} {"":<12} {"S1":>4} {"S2":>4} {"S3":>4} {"S4":>4} {"":>5} {"AI?":>4}')
print('=' * 72)

for idx, row in enumerate(rows, 1):
 ts, name, sid, phone, line = row[0], row[1], row[2], row[3], row[4]
 s1, s2, s3, s4, total = row[5], row[6], row[7], row[8], row[9]
 ai = '⚠️' if row[10] == 'True' else '✅'
 print(f'{idx:>3} {name:<20} {sid:<12} {s1:>4} {s2:>4} {s3:>4} {s4:>4} {total:>5} {ai:>4}')

# 
if rows:
 scores = [float(r[9]) for r in rows]
 print(f'\n📈 : ={min(scores):.1f}, ={max(scores):.1f}, ={sum(scores)/len(scores):.1f}')

## 💾 Download Score Files


In [ ]:
# CSV ( Google Colab)
try:
 from google.colab import files
 files.download(SCORES_CSV)
 print(f'✅ {SCORES_CSV}')
except:
 print(f'📄 : {os.path.abspath(SCORES_CSV)}')